#  Capítulo 6: Percolação em Meio Poroso
## Modelagem Numérica da Infiltração Vertical em Regime Permanente

### 🎯 Objetivos de Aprendizagem
1. Compreender a redução da equação de Richards para regime permanente.
2. Implementar o modelo de van Genuchten-Mualem para retenção e condutividade.
3. Discretizar e resolver numericamente o perfil de potencial matricial $\psi(z)$.
4. Interpretar fisicamente os perfis de umidade $\theta(z)$ e pressão no solo.

> 💡 **Nota Pedagógica**: Este notebook acompanha o Capítulo 6 do livro. Os comentários marcados com `` e `🔍` guiam seu raciocínio físico e numérico. Não pule as etapas de verificação dimensional!

### 🔧 1. Importações e Configurações

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuração de estilo para visualização técnica
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (8, 12)

### 📦 2. Parâmetros Físicos do Solo e Fluxo
Definimos as propriedades do solo franco-arenoso e as condições de contorno conforme o estudo de caso do livro.

In [ ]:
# === PARÂMETROS DO SOLO (van Genuchten-Mualem) ===
theta_r = 0.058      # Conteúdo de água residual [m³/m³]
theta_s = 0.410      # Conteúdo de água na saturação [m³/m³]
alpha = 7.4          # Inverso da pressão de entrada de ar [m⁻¹]
n = 1.56             # Parâmetro de distribuição de poros [-]
m = 1.0 - 1.0/n      # Parâmetro derivado [-]
K_s = 5.1e-5         # Condutividade hidráulica saturada [m/s]
L_param = 0.5        # Conectividade de poros [-]

# === CONDIÇÕES DE CONTORNO E GEOMETRIA ===
q0 = 1.0e-6          # Fluxo vertical descendente imposto [m/s] (≈ 86,4 mm/dia)
L = 3.0              # Profundidade do lençol freático [m]
N = 300              # Número de nós na discretização vertical
dz = L / N           # Espaçamento vertical [m]

# 🔍 Verificação física inicial:
# Em regime permanente descendente, o fluxo imposto q0 não pode exceder K_s.
# Se q0 >= K_s, o solo saturaria e a formulação não-saturada perderia validade.
assert q0 < K_s, "❌ Fluxo imposto maior que K_s! O solo não consegue drenar q0 em regime não-saturado."
print(f"✅ Verificação: q0 = {q0:.2e} m/s < K_s = {K_s:.2e} m/s. Condição física válida.")

### 📐 3. Modelo de van Genuchten-Mualem
Implementamos as funções constitutivas $\theta(\psi)$ e $K(\psi)$ que fecham a equação de Richards.

>  **Dica**: Note que o modelo usa $|\psi|$ porque a sucção é sempre positiva na zona não-saturada. O código protege contra divisões por zero e garante limites físicos ($\theta_r \leq \theta \leq \theta_s$).

In [ ]:
def theta_psi(psi, params):
    """Retenção de água: θ(ψ) [m³/m³]
    Entrada: psi [m] (potencial matricial, negativo ou zero)
    Saída: theta [m³/m³]
    """
    abs_psi = np.abs(psi)
    denom = (1.0 + (params['alpha'] * abs_psi)**params['n'])**params['m']
    theta = params['theta_r'] + (params['theta_s'] - params['theta_r']) / denom
    # Proteção numérica: θ não pode sair do intervalo físico
    return np.clip(theta, params['theta_r'], params['theta_s'])

def K_psi(psi, params):
    """Condutividade hidráulica: K(ψ) [m/s]
    Entrada: psi [m]
    Saída: K [m/s]
    """
    theta = theta_psi(psi, params)
    Se = (theta - params['theta_r']) / (params['theta_s'] - params['theta_r'])
    # Evita divisão por zero em solos muito secos
    Se = np.maximum(Se, 1e-12)
    term1 = Se**params['L_param']
    term2 = (1.0 - (1.0 - Se**(1.0/params['m']))**params['m'])**2
    return params['K_s'] * term1 * term2

# Agrupa parâmetros em dicionário para facilitar passagem
params = {
    'theta_r': theta_r, 'theta_s': theta_s,
    'alpha': alpha, 'n': n, 'm': m,
    'K_s': K_s, 'L_param': L_param
}

### 🔢 4. Discretização e Solução Numérica
A equação de Richards em regime permanente ($\partial\theta/\partial t = 0$) e fluxo constante $q = -q_0$ reduz-se a:
$$ \frac{d\psi}{dz} = \frac{q_0}{K(\psi)} - 1 $$

Integramos do lençol ($z=-L, \psi=0$) até a superfície ($z=0$) usando um esquema iterativo tipo Picard, conforme descrito no livro.

In [ ]:
# === DISCRETIZAÇÃO ESPACIAL ===
z = np.linspace(-L, 0, N+1)  # Coordenada vertical [m]
psi = np.zeros_like(z)       # Inicializa potencial matricial
theta = np.zeros_like(z)     # Inicializa umidade

# Condição de contorno no lençol freático
psi[0] = 0.0
theta[0] = theta_s  # Solo saturado na base

# === ESQUEMA NUMÉRICO (Picard Iterativo) ===
# Marchamos da base (z=-L) para a superfície (z=0)
tol = 1e-6
max_iter = 50

for i in range(1, N+1):
    # Chute inicial: mantém o valor do nó anterior
    psi_guess = psi[i-1]
    
    for _ in range(max_iter):
        K_prev = K_psi(psi_guess, params)
        psi_new = psi[i-1] + dz * (q0 / K_prev - 1.0)
        
        # Critério de convergência local
        if np.abs(psi_new - psi_guess) < tol:
            break
        psi_guess = psi_new
        
    psi[i] = psi_guess
    theta[i] = theta_psi(psi[i], params)

print("✅ Solução numérica concluída.")
print(f"   ψ na superfície: {psi[-1]:.3f} m")
print(f"   θ na superfície: {theta[-1]:.3f} m³/m³")

### 📊 5. Visualização e Interpretação Física
Plote os perfis e responda às perguntas guiadas abaixo.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7, 9))

# Perfil de potencial matricial
ax1.plot(psi, z, 'b-', linewidth=2)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_ylabel('Profundidade z [m]', fontsize=13)
ax1.set_xlabel('Potencial Matricial ψ [m]', fontsize=13)
ax1.set_title('Perfil de Pressão no Solo (Solução da Eq. de Richards)', fontsize=14)
ax1.grid(True, linestyle=':', alpha=0.7)
ax1.invert_yaxis()  # Profundidade cresce para baixo

# Perfil de umidade
ax2.plot(theta, z, 'g-', linewidth=2)
ax2.axvline(x=theta_r, color='red', linestyle='--', label='θ residual')
ax2.axvline(x=theta_s, color='orange', linestyle='--', label='θ saturação')
ax2.set_xlabel('Conteúdo de Umidade θ [m³/m³]', fontsize=13)
ax2.set_ylabel('Profundidade z [m]', fontsize=13)
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.7)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

###  6. Análise Guiada (Para o Estudante)
Responda em seu caderno ou em uma nova célula de markdown:

1. **Por que $\psi$ se torna mais negativo à medida que subimos em direção à superfície?**
   > 💡 *Pista*: Relacione com o balanço entre força gravitacional ($-1$) e resistência do meio ($q_0/K$).

2. **O que aconteceria se dobrássemos o fluxo imposto $q_0$ para $2\times 10^{-6}$ m/s?**
   > 💡 *Pista*: O solo ficaria mais úmido ou mais seco? Como $K(\psi)$ reage?

3. **Verifique a consistência dimensional da Eq. (6.8):**
   $$ \frac{d\psi}{dz} = \frac{q_0}{K(\psi)} - 1 $$
   >  *Unidades*: $[d\psi/dz] = [-]$, $[q_0/K] = [m/s]/[m/s] = [-]$, $[1] = [-]$. Tudo adimensionalizado corretamente.

4. **Limites físicos:** Se $z \to -L$ (lençol), $\psi \to 0$ e $K \to K_s$. O termo $q_0/K \to q_0/K_s < 1$. Logo, $d\psi/dz < 0$. Isso condiz com o gráfico?

### 🚀 7. Exercícios Práticos
Copie as células abaixo e modifique o código conforme solicitado.

#### 📘 Nível Graduação
**Exercício A:** Altere o tipo de solo para **argila** ($K_s = 1.0\times 10^{-7}$ m/s, $\alpha=1.5$, $n=1.3$, $\theta_r=0.08$, $\theta_s=0.48$). Mantendo $q_0$ inalterado, o solo consegue sustentar esse fluxo? O que acontece com $\psi$ na superfície?

####  Nível Pós-Graduação
**Exercício B (Sensibilidade):** Implemente um laço que varie $q_0$ de $10^{-7}$ a $10^{-5}$ m/s e plotte $\psi_{superfície}$ vs $q_0$. Identifique o ponto onde $\psi \to -\infty$ (limite de fluxo máximo sustentável em regime não-saturado).

In [ ]:
# Espaço para seu código (Exercício A)
# Dica: Basta redefinir os parâmetros e rodar novamente a célula de solução (Seção 4).

params_argila = {
    'theta_r': 0.08, 'theta_s': 0.48,
    'alpha': 1.5, 'n': 1.3, 'm': 1.0 - 1.0/1.3,
    'K_s': 1.0e-7, 'L_param': 0.5
}

# TODO: Implemente a simulação para argila aqui ou copie/adapte as células anteriores.

### 📚 Referências do Notebook
- van Genuchten, M. Th. (1980). A closed-form equation for predicting the hydraulic conductivity of unsaturated soils. *SSSAJ*, 44(5), 892-898.
- Livro-texto: **Capítulo 6** (Equação de Richards, Modelo de van Genuchten, Fluxo Vertical Permanente).
- Código-fonte validado: `https://github.com/JaderLugon/fenomenos-transporte-livro`